# Week 12 — BBO capstone driver

Round 12. Make the line search quantitative. Three points on a line define a parabola, so fit one per function and solve for the vertex rather than probing another λ by hand.

F4's fitted vertex sits at λ* ≈ 0.883 and the query goes to λ = 0.881. F8's at λ* ≈ 0.518, queried at 0.6. F7 confirms its vertex at λ ≈ 0.866. F6 takes the vertex plus a +0.15 lift on x2 to separate gradient from curvature on that axis.

**A gap I notice this round and cannot fix cheaply:** every F7 point ever submitted is collinear. Five of its six dimensions have received no information at all, across five consecutive rounds. The line search bought efficiency by discarding the perpendicular directions, and that bill comes due in W13.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 12
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 12
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: 'collinear node hunt',
    2: 'perpendicular offset test',
    3: 'new perturbation',
    4: 'line vertex λ*≈0.881',
    5: '8 steps along ray',
    6: 'vertex + x2 lift',
    7: 'confirm vertex λ≈0.866',
    8: 'line λ=0.6 toward vertex',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 11. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — quadratic vertex fits

`bbo.fit_line_vertex` fits ax²+bx+c through the collinear points and returns −b/2a. A non-concave fit returns NaN, meaning the line has no interior maximum.

In [ ]:
proposals = {
    1: np.array([0.270105, 0.311087]),
    2: np.array([0.575432, 0.810899]),
    3: np.array([0.167485, 0.838106, 0.513161]),
    4: np.array([0.462508, 0.448451, 0.44279, 0.547953]),
    5: np.array([0.252806, 0.786874, 0.031998, 0.717917]),
    6: np.array([0.302252, 0.664134, 0.708267, 0.649836, 0.375274]),
    7: np.array([0.562174, 0.453772, 0.511847, 0.541027, 0.446492, 0.526734]),
    8: np.array([0.315593, 0.410194, 0.372771, 0.439121, 0.621278, 0.393905, 0.674416, 0.337825]),
}

# Fit the vertex for F4 from its three collinear points (W8 lam=0, W11 lam=0.7, and the
# W12 query itself), then confirm the submitted point sits on that line.
anchor4 = np.array(bbo.HISTORY[8][4][0])
lam_submitted = bbo.project_to_line(anchor4, proposals[4])
print(f"F4 submitted at lambda = {lam_submitted:.4f}")
print(f"F4 on the centre line: {np.allclose(bbo.centre_line(anchor4, lam_submitted), proposals[4], atol=1e-5)}")

# How much perpendicular information does each function actually have?
rows = []
for fid in bbo.FUNC_IDS:
    X, _, _ = bbo.load(fid, up_to=PRIOR)
    rank = int(np.linalg.matrix_rank(X - X.mean(0), tol=1e-6)) if len(X) > 1 else 0
    rows.append(dict(func=f"F{fid}", d=bbo.DIMS[fid], n_points=len(X),
                     spanned_rank=rank, unexplored_dims=bbo.DIMS[fid]-rank))
pd.DataFrame(rows)


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.669937, 0.751452],
    2: [0.354432, 0.449899],
    3: [0.111275, 0.774316, 0.516951],
    4: [0.183872, 0.065353, 0.017618, 0.904326],
    5: [0.308806, 0.730874, 0.091998, 0.661917],
    6: [0.070021, 0.530732, 0.952853, 0.825805, 0.228797],
    7: [0.965568, 0.153915, 0.588691, 0.807159, 0.099427, 0.700138],
    8: [0.038983, 0.275485, 0.181927, 0.347803, 0.803194, 0.234763, 0.936041, 0.094563],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 12 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: -1.1805e-25,
#     2: 0.037313,
#     3: -0.047922,
#     4: -2.874597,
#     5: 24.903137,
#     6: -0.948038,
#     7: 0.53412,
#     8: 9.094693,
# }
#
# Four new bests: F4 -2.875, F5 24.903, F7 0.534, F8 9.095.
# F4's vertex prediction was accurate - lambda* = 0.883 fitted, queried at 0.881,
# returned the best F4 value of the campaign. The quadratic fit works.
# F5 is still growing faster than any log-quadratic fit predicts; x3 hits the domain
# floor at roughly n=12 steps, so there is about one step of headroom left.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
